<a href="https://colab.research.google.com/github/kmmmm25/KumaGPT/blob/main/Kuma_GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")

save_dir = "/content/drive/MyDrive/KumaGPT/check"

Mounted at /content/drive


In [1]:
class Tokenizer:
    def __init__(self, chars: list[str]) -> None:
        self.str_to_idx: dict[str, int] = dict()

        self.str_to_idx["<|endoftext|>"] = 0
        self.str_to_idx["<user>"] = 1
        self.str_to_idx["<assistant>"] = 2
        # utf-8
        for i in range(256):
            if f'<utf8_{i}>' not in self.str_to_idx:
                self.str_to_idx[f'<utf8_{i}>'] = len(self.str_to_idx)
        for char in chars:
            self.str_to_idx[char] = len(self.str_to_idx) if char not in self.str_to_idx else self.str_to_idx[char]

        # 登録したIDに重複がないか確認
        assert len(self.str_to_idx.values()) == len(set(self.str_to_idx.values()))

        self.idx_to_str: dict[int, str] = dict()
        for key, value in self.str_to_idx.items():
            self.idx_to_str[value] = key

    def encode(self, text: str, eot=False) -> list[int]:
        result = []

        special_tokens = [
            "<|endoftext|>",
            "<user>",
            "<assistant>",
        ]

        i = 0

        while i < len(text):

            # special tokenか確認
            matched = False

            for special_token in special_tokens:
                if text.startswith(special_token, i):
                    result.append(
                        self.str_to_idx[special_token]
                    )

                    i += len(special_token)
                    matched = True
                    break

            if matched:
                continue

            # 普通の文字
            char = text[i]

            maybe_token = self.str_to_idx.get(char)

            if maybe_token is not None:
                result.append(maybe_token)

            else:
                utf_8_num = list(char.encode("utf-8"))

                for num in utf_8_num:
                    result.append(
                        self.str_to_idx[f"<utf8_{num}>"]
                    )

            i += 1

        if eot:
            result.append(
                self.str_to_idx["<|endoftext|>"]
            )

        return result

    def decode(self, tokens: list[int]) -> str:
        decoded_with_utf_token: list[str] = [self.idx_to_str[token] for token in tokens]
        decoded_postprocess_utf: list[str] = []
        utf_tokens: list[int] = []
        for token in decoded_with_utf_token:
            if token.startswith("<utf8_"):
                utf_num = int(token.replace("<utf8_", "").replace(">", ""))
                utf_tokens.append(utf_num)
            else:
                if utf_tokens:
                    decoded_postprocess_utf.append(bytes(utf_tokens).decode("utf-8", errors="replace"))
                    utf_tokens = []
                decoded_postprocess_utf.append(token)
        if utf_tokens:
            decoded_postprocess_utf.append(bytes(utf_tokens).decode("utf-8", errors="replace"))
            utf_tokens = []
        return "".join(decoded_postprocess_utf)

    def decode_with_utf(self, tokens:list[int]) -> str:
        return "".join([self.idx_to_str[token] for token in tokens])

In [2]:
!pip install -q datasets

In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "wikimedia/wikipedia",
    "20231101.ja",
    split="train",
    streaming=True
)

dataset = dataset.shuffle(
    seed=42,
    buffer_size=10_000
)

README.md:   0%|          | 0.00/131k [00:00<?, ?B/s]

In [4]:
train_texts = []
val_texts = []
count = 0

# Iterate through the first dataset (IVUL-KAUST/MOLE)
for item in dataset:
  if count < 95000:
    if "text" in item:
      train_texts.append(item["text"])
  elif count < 100000:
    if "text" in item:
        val_texts.append(item["text"])
  else:
    break

  count += 1

all_text = "<|endoftext|>".join(train_texts)

In [ ]:
import sentencepiece as spm

VOCAB_SIZE = 8000

# すでに1本の巨大な文字列になっている想定
with open("tokenizer_train.txt", "w", encoding="utf-8") as f:
    f.write(all_text)

spm.SentencePieceTrainer.train(
    input="tokenizer_train.txt",
    model_prefix="kumagpt_unigram",
    vocab_size=VOCAB_SIZE,
    model_type="unigram",
    character_coverage=0.9995,
    user_defined_symbols=[
        "<user>",
        "<assistant>",
        "<|endoftext|>",
    ],
)

tokenizer = spm.SentencePieceProcessor(
    model_file="kumagpt_unigram.model"
)

print("vocab size:", tokenizer.get_piece_size())

In [ ]:
vocab = sorted(set(all_text))

tokenizer_before = Tokenizer(vocab)

vocab_size = len(tokenizer.str_to_idx)

print(vocab_size)

10890


# Transformer

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(2005)

class Attention(nn.Module):
  def __init__(self, d_model, d_head, h):
    assert d_head % h == 0

    super().__init__()

    self.Wk = nn.Linear(d_model, d_head)
    self.Wq = nn.Linear(d_model, d_head)
    self.Wv = nn.Linear(d_model, d_head)

    self.h = h

    self.linear = nn.Linear(d_head, d_model)

  def forward(self, x):

    k = self.Wk(x)
    q = self.Wq(x)
    v = self.Wv(x)

    B, T, d_head = k.shape

    k = k.view(B, T, self.h, d_head // self.h).transpose(1, 2)
    q = q.view(B, T, self.h, d_head // self.h).transpose(1, 2)
    v = v.view(B, T, self.h, d_head // self.h).transpose(1, 2)

    y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

    # (B, h, T, d_head)
    #          ↓
    # (B, T, h, d_head)
    y = y.transpose(1, 2)

    attention = y.reshape(B, T, d_head)

    output = self.linear(attention)

    return output

class Block(nn.Module):
  def __init__(self, d_model, d_head, d_ff, h):
    super().__init__()
    self.ln1 = nn.LayerNorm(d_model)
    self.attn = Attention(d_model, d_head, h)

    self.ln2 = nn.LayerNorm(d_model)
    self.ff = nn.Sequential(
        nn.Linear(d_model, d_ff),
        nn.GELU(),
        nn.Linear(d_ff, d_model)
    )

    self.dropout = nn.Dropout(0.1)

  def forward(self, x):
    x = x + self.dropout(self.attn(self.ln1(x)))
    x = x + self.dropout(self.ff(self.ln2(x)))

    return x


class Kuma_GPT(nn.Module):
  def __init__(self, vocab_size, block_size, d_model, d_head, d_ff, n_layer, h):
    super().__init__()
    self.token_embedding = nn.Embedding(vocab_size, d_model)
    self.pos_embedding = nn.Embedding(block_size, d_model)

    self.blocks = nn.ModuleList([
        Block(d_model, d_head, d_ff, h)
        for _ in range(n_layer)
    ])

    self.lnf = nn.LayerNorm(d_model)
    self.linear_output = nn.Linear(d_model, vocab_size)

    self.block_size = block_size

  def forward(self, input, targets=None):
    B, T = input.shape

    tok_emb = self.token_embedding(input)

    pos_input = torch.arange(T, device=input.device)
    pos_emb = self.pos_embedding(pos_input)

    x = tok_emb + pos_emb

    for block in self.blocks:
      x = block(x)

    y = self.lnf(x)
    logits = self.linear_output(y)

    loss = None
    if targets is not None:
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-1)

    return logits, loss

  @torch.no_grad()
  def generate(self, input, max, temperature=0.7, top_k=30):
    user_id = tokenizer.str_to_idx["<user>"]
    assistant_id = tokenizer.str_to_idx["<assistant>"]
    eot_id = tokenizer.str_to_idx["<|endoftext|>"]

    stop_tokens = {
        user_id: "<user>",
        eot_id: "<|endoftext|>",
    }

    self.eval()
    results = input.clone()

    for _ in range(max):
        x = results[:, -self.block_size:]
        logits, _ = self(x)

        next_logits = logits[:, -1, :] / temperature

        if top_k is not None:
            values, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
            next_logits[next_logits < values[:, [-1]]] = -float("inf")

        probs = F.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        next_token_id = next_token.item()

        if next_token_id in stop_tokens:
          print(
              f"[generate stop] special token generated: "
              f"{stop_tokens[next_token_id]} "
              f"(id={next_token_id})"
          )
          break

        results = torch.cat([results, next_token], dim=1)

    return results

parameter初期化、データ整備

In [ ]:
block = 256
d_model = 256
d_ff = d_model * 4
d_head = d_model
n_layer = 6
h = 4

In [ ]:
train = train_texts[:]

print(len(train_texts))

train_tokens = []
for x in train:
  tokens = tokenizer.encode(x, eot=True)
  train_tokens.append(tokens)


val = val_texts[:]

print(len(val_texts))

val_tokens = []
for x in val:
  tokens = tokenizer.encode(x)
  val_tokens.append(tokens)

95000
5000


In [ ]:
import math

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


train_flat_tokens = [token for sublist in train_tokens for token in sublist]

batch_size = 32

chunk_size = block + 1
num_chunks = len(train_flat_tokens) // chunk_size
train_tokens = torch.tensor(train_flat_tokens[:num_chunks * chunk_size]).view(num_chunks, chunk_size)

val_flat_tokens = [token for sublist in val_tokens for token in sublist]

chunk_size = block + 1
num_chunks = len(val_flat_tokens) // chunk_size
val_tokens = torch.tensor(val_flat_tokens[:num_chunks * chunk_size]).view(num_chunks, chunk_size)

epoch = 20


steps_per_epoch = math.ceil(len(train_tokens) / batch_size)
total_steps = steps_per_epoch * epoch

warmup_steps = 500


# 学習再開用　pretrained_base変更必須


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 読み込みたいcheckpoint
checkpoint_path = "/content/drive/MyDrive/KumaGPT/check/pretrained_base4_19.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)


model = Kuma_GPT(
    vocab_size=vocab_size,
    block_size=block,
    d_model=d_model,
    d_head=d_head,
    d_ff=d_ff,
    n_layer=n_layer,
    h=h,
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3
)

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.01,   # 最初は peak_lr の1%
    end_factor=1.0,
    total_iters=500
)

cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_steps - warmup_steps,
    eta_min=1e-5
)

scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        warmup_scheduler,
        cosine_scheduler
    ],
    milestones=[warmup_steps]
)

scheduler.load_state_dict(
    checkpoint["scheduler_state_dict"]
)

optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

saved_epoch = checkpoint["epoch"]

start_epoch = saved_epoch + 1

print("checkpoint loaded")
print("saved epoch:", saved_epoch)
print("resume epoch:", start_epoch)

print("optimizer lr:", optimizer.param_groups[0]["lr"])
print("scheduler last_epoch:", scheduler.last_epoch)
print("scheduler last lr:", scheduler.get_last_lr())

checkpoint loaded
saved epoch: 19
resume epoch: 20
optimizer lr: 1e-05
scheduler last_epoch: 391540
scheduler last lr: [1e-05]


# 初期学習用

In [ ]:
model = Kuma_GPT(vocab_size, block, d_model, d_head, d_ff, n_layer, h).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-4)

warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer,
    start_factor=0.01,   # 最初は peak_lr の1%
    end_factor=1.0,
    total_iters=500
)

cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=total_steps - warmup_steps,
    eta_min=1e-5
)

scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer,
    schedulers=[
        warmup_scheduler,
        cosine_scheduler
    ],
    milestones=[warmup_steps]
)

start_epoch = 0

# **学習**

In [ ]:
save_dir = "/content/drive/MyDrive/KumaGPT/check"

for i in range(start_epoch, epoch):
    #train
    model.train()

    perm = torch.randperm(len(train_tokens))
    shuffled_train_tokens = train_tokens[perm]

    total_train_loss = 0
    num_batches = 0
    for j in range(0, len(shuffled_train_tokens), batch_size):

      batch_token = shuffled_train_tokens[j:j+batch_size].to(device)

      input = batch_token[:, :-1]
      target = batch_token[:, 1:]

      optimizer.zero_grad()

      logits, loss = model(input, target)

      loss.backward()
      optimizer.step()

      total_train_loss += loss.item()
      num_batches += 1

      scheduler.step()

    train_loss = total_train_loss / num_batches

    #val
    model.eval()

    total_val_loss = 0
    num_batches = 0

    with torch.no_grad():
      for j in range(0, len(val_tokens), batch_size):

        batch_token = val_tokens[j:j+batch_size].to(device)

        input = batch_token[:, :-1]
        target = batch_token[:, 1:]

        logits, loss = model(input, target)

        total_val_loss += loss.item()
        num_batches += 1

    val_loss = total_val_loss / num_batches

    #生成
    kaiwa = '日本の首都である東京は'
    sentence_token = tokenizer.encode(kaiwa)

    x = torch.tensor(sentence_token, dtype=torch.long).unsqueeze(0).to(device)

    y = model.generate(x, 100)

    gen = tokenizer.decode(y[0].tolist())


    #保存
    config = {
        "vocab_size": vocab_size,
        "block_size": block,
        "d_model": d_model,
        "d_head": d_head,
        "d_ff": d_ff,
        "n_layer": n_layer,
        "h": h,
    }

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),

        "epoch": i,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "generated_prompt": gen,

        "config": config,
        "tokenizer_str_to_idx": tokenizer.str_to_idx,
      }

    save_path = f"{save_dir}/pretrained_base4_{i}.pt"

    torch.save(checkpoint, save_path)

    print(f"保存完了: {save_path}")
    print("")
    print(gen)
    print("")


    print(f'epoch {i}')

    print(f'train loss: {train_loss:.6f}')
    print(f'val loss: {val_loss:.6f}')

    print('-------------------------------------------------------------------------------------------------------')

保存完了: /content/drive/MyDrive/KumaGPT/check/pretrained_base4_4.pt

日本の首都である東京は、アメリカ合衆国上院である。

概要 
アメリカ合衆国の首都のシンキロ・アイアン・ランドである。

アメリカ合衆国の最も有名なアメリカ合衆国からの輸出と輸出の際、イギリスの貿易には、米国における輸出に

epoch 4
train loss: 2.342604
val loss: 2.302243
-------------------------------------------------------------------------------------------------------
保存完了: /content/drive/MyDrive/KumaGPT/check/pretrained_base4_5.pt

日本の首都である東京は、日本の最初の首都であった。

1969年、日本の首都である株式会社日本の首都である日本大学との関係を結んだ。

2002年、日本においても、東京・日本首都圏の首都圏であった上京の日本首都圏を経営して

epoch 5
train loss: 2.319903
val loss: 2.284746
-------------------------------------------------------------------------------------------------------
保存完了: /content/drive/MyDrive/KumaGPT/check/pretrained_base4_6.pt

日本の首都である東京は、新東京にある北京市の南京地下鉄にある東京都港区の中央駅である東京駅発着の東京駅と東京駅発着の東京駅と、東京駅発着の東京駅発着の東京駅の上り線列車が通行可能。

東京駅 
 東京駅 ： 東京駅 - 東

epoch 6
train loss: 2.301392
val loss: 2.268089
-------------------------------------------------------------------------------------------------------
保存完了: /

In [ ]:
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Batch size: {batch_size}")
print(f"Block size: {block}")
print(f"Tokens / step: {batch_size}")
print(f"Total training tokens: {len(train_flat_tokens)}")

Parameters: 10,391,178
Batch size: 32
Block size: 256
Tokens / step: 32
Total training tokens: 161000558


In [ ]:
sentence = "なぜこのような"
sentence_token = tokenizer.encode(sentence)

x = torch.tensor(sentence_token, dtype=torch.long).unsqueeze(0).to(device)

y = model.generate(x, 50)

print(tokenizer.decode(y[0].tolist()))

なぜこのような状況を確認しているかを知る。この状況は、「このシール」を用いて「このシールは、今日のシールにいるが、


In [ ]:
print(os.path.exists(save_path))
print(f"{os.path.getsize(save_path) / 1024**2:.2f} MB")

True
119.21 MB


In [ ]:
loaded_checkpoint = torch.load(
    save_path,
    map_location=device,
    weights_only=False
)

print(loaded_checkpoint.keys())
print("保存epoch:", loaded_checkpoint["epoch"])
print("validation loss:", loaded_checkpoint["val_loss"])

dict_keys(['model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'epoch', 'train_loss', 'val_loss', 'generated_prompt', 'config', 'tokenizer_str_to_idx'])
保存epoch: 19
validation loss: 2.17678829484623


# Full Fine Tuning

In [ ]:
model.eval()

user = "<user>"
assistant = "<assistant>"

kaiwa = '日本の首都である東京は'
sentence_token = tokenizer.encode(kaiwa)

x = torch.tensor(sentence_token, dtype=torch.long).unsqueeze(0).to(device)

y = model.generate(x, 100)

print(tokenizer.decode(y[0].tolist()))

日本の首都である東京は日本国内で1年間の最高峰である。

概要 
「東京湾における最高峰」としては、2005年（平成17年）5月に開催された第6回世界大会（日本）では、2005年（平成17年）9月に開催された第7回世界大会


16epoch学習済みKumaGPT

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 読み込みたいcheckpoint
checkpoint_path = "/content/drive/MyDrive/KumaGPT/checkpoints/pretrained_base2_18.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
)


model = Kuma_GPT(
    vocab_size=vocab_size,
    block_size=block,
    d_model=d_model,
    d_head=d_head,
    d_ff=d_ff,
    n_layer=n_layer,
    h=h,
).to(device)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

print("checkpoint loaded")

In [ ]:
from datasets import load_dataset

dataset = load_dataset('llm-jp/oasst1-21k-ja', split='train')

print(dataset['conversations'][4][0])
print(dataset['conversations'][4][0]['from'])

README.md:   0%|          | 0.00/724 [00:00<?, ?B/s]

oasst1-21k-ja.jsonl: reconstructing file:   0%|          |  0.00B / 42.0MB            

oasst1-21k-ja.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/21164 [00:00<?, ? examples/s]

{'from': 'human', 'value': '「速く走るためにゆっくり走る」という言葉を、長距離走と持久力トレーニングの文脈で説明してください。また、科学的根拠の参考文献も添えてください。'}
human


In [ ]:
def apply_chat_template(example):
  formatted_messages = []
  for message in example['conversations']:
    role = message['from']
    value = message['value'].strip()

    if role == "human":
      formatted_messages.append(f"<user>{value}")

    elif role == "gpt":
      formatted_messages.append(f"<assistant>{value}<|endoftext|>")

  text = "\n".join(formatted_messages)

  return {"text": text}

In [ ]:
formatted_dataset = dataset.map(
    apply_chat_template,
    remove_columns=dataset.column_names
)

Map:   0%|          | 0/21164 [00:00<?, ? examples/s]

In [ ]:
print(formatted_dataset)
print(formatted_dataset[1]['text'])
print(len(formatted_dataset['text']))

Dataset({
    features: ['text'],
    num_rows: 21164
})
<user>スリーボディとは？
<assistant>もっと詳しく説明してください。<|endoftext|>
<user>物理学
<assistant>物理学における3体問題とは、3つの点質量の初期位置と速度がわかっているときに、その後の位置と速度を求める問題である。三体問題のすべての事例を解くことのできる閉形式は存在しないので、数値的手法によって解かれることが多い。<|endoftext|>
21164


In [ ]:
# 21164 x 0.9 = 19047.6

train = formatted_dataset[:19047]['text']

print(len(train))

train_tokens = []
for x in train:
  tokens = tokenizer.encode(x)
  train_tokens.append(tokens)

val = formatted_dataset[19047:]['text']

print(len(val))

val_tokens = []
for x in val:
  tokens = tokenizer.encode(x)
  val_tokens.append(tokens)

19047
2117


In [ ]:
end_text = "<|endoftext|>"
end_id = tokenizer.str_to_idx[end_text]

test_tokens = tokenizer.encode(end_text)

print("特殊トークンID:", end_id)
print("encode結果:", test_tokens)
print("トークン数:", len(test_tokens))

特殊トークンID: 0
encode結果: [0]
トークン数: 1


In [ ]:
user = "<user>"
assistant = "<assistant>"

In [ ]:
train_flat_tokens = [token for sublist in train_tokens for token in sublist]
num_chunks = len(train_flat_tokens) // chunk_size
train_tokens = torch.tensor(train_flat_tokens[:num_chunks * chunk_size]).view(num_chunks, chunk_size)



val_flat_tokens = [token for sublist in val_tokens for token in sublist]
num_chunks = len(val_flat_tokens) // chunk_size
val_tokens = torch.tensor(val_flat_tokens[:num_chunks * chunk_size]).view(num_chunks, chunk_size)

epoch = 20


sft_optimizer = torch.optim.AdamW(model.parameters(), lr=4e-3)

steps_per_epoch = math.ceil(len(train_tokens) / batch_size)
total_steps = steps_per_epoch * epoch

warmup_steps = 500

sft_warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    sft_optimizer,
    start_factor=0.01,   # 最初は peak_lr の1%
    end_factor=1.0,
    total_iters=500
)

sft_cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    sft_optimizer,
    T_max=total_steps - warmup_steps,
    eta_min=1e-5
)

sft_scheduler = torch.optim.lr_scheduler.SequentialLR(
    sft_optimizer,
    schedulers=[
        sft_warmup_scheduler,
        sft_cosine_scheduler
    ],
    milestones=[warmup_steps]
)

for i in range(epoch):
    #train
    model.train()

    total_train_loss = 0
    num_batches = 0
    for j in range(0, len(train_tokens), batch_size):

      batch_token = train_tokens[j:j+batch_size].to(device)

      input = batch_token[:, :-1]
      target = batch_token[:, 1:]

      sft_optimizer.zero_grad()

      logits, loss = model(input, target)

      loss.backward()
      sft_optimizer.step()

      total_train_loss += loss.item()
      num_batches += 1

    sft_scheduler.step()

    train_loss = total_train_loss / num_batches

    #val
    model.eval()

    total_val_loss = 0
    num_batches = 0

    with torch.no_grad():
      for j in range(0, len(val_tokens), batch_size):

        batch_token = val_tokens[j:j+batch_size].to(device)

        input = batch_token[:, :-1]
        target = batch_token[:, 1:]

        logits, loss = model(input, target)

        total_val_loss += loss.item()
        num_batches += 1

    val_loss = total_val_loss / num_batches


    print(f'epoch {i + 19}')

    print(f'train loss: {train_loss:.6f}')
    print(f'val loss: {val_loss:.6f}')

    print('')


    prompt = [
        '今日の天気は何ですか？',
        '犬ってどんな動物？',
        '疲れたので、少し励ましてください',
        ]

    for kaiwa in prompt:
      sentence = user + kaiwa + '\n' + assistant
      sentence_token = tokenizer.encode(sentence)

      x = torch.tensor(sentence_token, dtype=torch.long).unsqueeze(0).to(device)

      y = model.generate(x, 50)

      print(tokenizer.decode(y[0].tolist()))
      print("")

    print("--------------------------------------------------------------------------------------------------------------------")


    #保存
    config = {
        "vocab_size": vocab_size,
        "block_size": block,
        "d_model": d_model,
        "d_head": d_head,
        "d_ff": d_ff,
        "n_layer": n_layer,
        "h": h,
    }

    checkpoint = {
        "model_state_dict": model.state_dict(),
        "sft_optimizer_state_dict": sft_optimizer.state_dict(),
        "sft_scheduler_state_dict": sft_scheduler.state_dict(),

        "epoch": i + 19,
        "train_loss": train_loss,
        "val_loss": val_loss,

        "config": config,
        "tokenizer_str_to_idx": tokenizer.str_to_idx,
      }

    save_path = f"{save_dir}/sft_model2_{i}.pt"

    torch.save(checkpoint, save_path)

epoch 19
train loss: 2.187056
val loss: 2.009542

<user>今日の天気は何ですか？
<assistant>これは、1970年代前半には、気候変動における気候変動の原因となっていた。この気候変動は、気候変動の

[generate stop] special token generated: <|endoftext|> (id=0)
<user>犬ってどんな動物？
<assistant>もちろん。犬っている犬ってないのか？

<user>疲れたので、少し励ましてください
<assistant>もちろん、彼が楽しいでしょう。この音楽は、この曲を使って、楽しく楽しく楽しんでください。

コーラス

--------------------------------------------------------------------------------------------------------------------
epoch 20
train loss: 2.046482
val loss: 1.939010

<user>今日の天気は何ですか？
<assistant>おそらくいいえ、これは人間があなたの意思を尊重しているのです。これは、人間が人間の意思を尊重し、人間

<user>犬ってどんな動物？
<assistant>私は何か？あなたがいないのか？

犬っ引き

犬っ引き

犬の犬っ引き

犬っ引き

犬っ引き

犬

<user>疲れたので、少し励ましてください
<assistant>もちろん、私のようなコスプレに興味があるのです。
彼らは、私のようなサンプルを取り、そのようなサンプ

--------------------------------------------------------------------------------------------------------------------
epoch 21
train loss: 1.983160
val loss: 1.891016

[generate stop] special token generated: <|endoftext|> (id=0)
<user>今日の天気は何ですか？
<assistant>あなた

In [ ]:
kaiwa = 'おはよう！'
sentence = user + kaiwa + '\n' + assistant
sentence_token = tokenizer.encode(sentence)

x = torch.tensor(sentence_token, dtype=torch.long).unsqueeze(0).to(device)

y = model.generate(x, 100)

print(tokenizer.decode(y[0].tolist()))

<user>おはよう！
<assistant>ダークカレット：ダークカレット：こんにちは。

ドワーフ：こんにちは！ダークカレットは、ダークカレットの世界にある。

マッサージ：こんにちは。

ドワーフ：こんにちは！ダークカレットは、ダークカレッ


In [ ]:
config = {
          "vocab_size": vocab_size,
          "block_size": block,
          "d_model": d_model,
          "d_head": d_head,
          "d_ff": d_ff,
          "n_layer": n_layer,
          "h": h,
      }

checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),

    "epoch": epoch,
    "train_loss": train_loss,
    "val_loss": val_loss,

    "config": config,
    "tokenizer_str_to_idx": tokenizer.str_to_idx,
}

save_path = f"{save_dir}/KumaGPT_2.pt"

torch.save(checkpoint, save_path)

print(f"保存完了: {save_path}")

保存完了: /content/drive/MyDrive/KumaGPT/check/KumaGPT_2.pt


In [ ]:
6